<a href="https://colab.research.google.com/github/YugeshwarV/Project_NASA_NEO/blob/main/NASA_Near_Earth_Object(NEO).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NASA Near-Earth Object (NEO) Tracking & Insights using Public API

### Step 1 & 2 : Getting Nasa Api Key & Extract Data Using NASA"s Asteroid API

In [ ]:
import requests
import json

# NASA API Key
API_KEY = "cAi3JqpTDM7Benl1ToMrDozdFxK63SBG3SE3Ykij"

# Begin with a start date (2024-01-01) and an end date 7 days later (2024-01-07).
start_date = "2024-01-01"
end_date = "2024-01-07"

# Construct API URL
url = f"https://api.nasa.gov/neo/rest/v1/feed?start_date={start_date}&end_date={end_date}&api_key={API_KEY}"

# Send GET request
response = requests.get(url)

# Check response
if response.status_code == 200:
    data = response.json()
    print("Data fetched successfully!")
else:
    print("Failed to fetch data:", response.status_code)

Data fetched successfully!


### To collect 10000 records

In [ ]:
import requests
import time
from datetime import datetime, timedelta

In [ ]:
API_KEY = "cAi3JqpTDM7Benl1ToMrDozdFxK63SBG3SE3Ykij"
BASE_URL = "https://api.nasa.gov/neo/rest/v1/feed"
START_DATE = datetime.strptime("2024-01-01", "%Y-%m-%d")
RECORD_LIMIT = 10000

# To format the date
def format_date(dt):
    return dt.strftime("%Y-%m-%d")

asteroid_data = []

# Loop through 7-day date range to get 10000 records
current_date = START_DATE
while len(asteroid_data) < RECORD_LIMIT:
    end_date = current_date + timedelta(days=6)
    url = f"{BASE_URL}?start_date={format_date(current_date)}&end_date={format_date(end_date)}&api_key={API_KEY}"

    response = requests.get(url)
    if response.status_code != 200:
        print("Error fetching data:", response.status_code)
        break

    data = response.json()

    # Flatten nested data
    near_earth_objects = data.get("near_earth_objects", {})
    for date in near_earth_objects:
        for obj in near_earth_objects[date]:
            asteroid_data.append(obj)
            if len(asteroid_data) >= RECORD_LIMIT:
                break
        if len(asteroid_data) >= RECORD_LIMIT:
            break

    print(f"Fetched {len(asteroid_data)} records up to {format_date(end_date)}")

    # Move to next 7 days
    current_date = end_date + timedelta(days=1)

print(f"Total records fetched: {len(asteroid_data)}")

Fetched 111 records up to 2024-01-07
Fetched 253 records up to 2024-01-14
Fetched 371 records up to 2024-01-21
Fetched 510 records up to 2024-01-28
Fetched 653 records up to 2024-02-04
Fetched 777 records up to 2024-02-11
Fetched 924 records up to 2024-02-18
Fetched 1039 records up to 2024-02-25
Fetched 1173 records up to 2024-03-03
Fetched 1305 records up to 2024-03-10
Fetched 1412 records up to 2024-03-17
Fetched 1551 records up to 2024-03-24
Fetched 1664 records up to 2024-03-31
Fetched 1803 records up to 2024-04-07
Fetched 1954 records up to 2024-04-14
Fetched 2109 records up to 2024-04-21
Fetched 2240 records up to 2024-04-28
Fetched 2359 records up to 2024-05-05
Fetched 2491 records up to 2024-05-12
Fetched 2599 records up to 2024-05-19
Fetched 2704 records up to 2024-05-26
Fetched 2818 records up to 2024-06-02
Fetched 2924 records up to 2024-06-09
Fetched 3019 records up to 2024-06-16
Fetched 3123 records up to 2024-06-23
Fetched 3229 records up to 2024-06-30
Fetched 3329 record

In [ ]:
START_DATE

datetime.datetime(2024, 1, 1, 0, 0)

In [ ]:
len(asteroid_data)

10000

### Step 3 - Data Cleaning Steps

In [ ]:
from datetime import datetime

# Use asteroid_data from previous step (assuming you import or load it here)
# For now, we assume asteroid_data is loaded
# Replace this line with actual import if using across notebooks
try:
    asteroid_data
except NameError:
    import json
    with open("asteroid_data.json", "r") as f:
        asteroid_data = json.load(f)

cleaned_asteroids = []
close_approaches = []

for obj in asteroid_data:
    try:
        # General asteroid info
        asteroid_id = int(obj.get("id"))
        name = obj.get("name")
        abs_mag = float(obj.get("absolute_magnitude_h", 0))

        diameter_min = obj["estimated_diameter"]["kilometers"]["estimated_diameter_min"]
        diameter_max = obj["estimated_diameter"]["kilometers"]["estimated_diameter_max"]

        is_hazardous = obj.get("is_potentially_hazardous_asteroid", False)

        cleaned_asteroids.append({
            "id": asteroid_id,
            "name": name,
            "absolute_magnitude_h": abs_mag,
            "estimated_diameter_min_km": diameter_min,
            "estimated_diameter_max_km": diameter_max,
            "is_potentially_hazardous_asteroid": is_hazardous
        })

        # Close approach data (can have multiple entries)
        for approach in obj.get("close_approach_data", []):
            date = datetime.strptime(approach.get("close_approach_date", ""), "%Y-%m-%d").date()
            rel_vel = float(approach["relative_velocity"]["kilometers_per_hour"])
            miss_km = float(approach["miss_distance"]["kilometers"])
            miss_ld = float(approach["miss_distance"]["lunar"])
            astro = float(approach["miss_distance"]["astronomical"])
            orbiting = approach.get("orbiting_body", "Earth")

            close_approaches.append({
                "neo_reference_id": asteroid_id,
                "close_approach_date": date,
                "relative_velocity_kmph": rel_vel,
                "astronomical": astro,
                "miss_distance_km": miss_km,
                "miss_distance_lunar": miss_ld,
                "orbiting_body": orbiting
            })

    except Exception as e:
        print("Error processing object:", obj.get("id"), e)

print(f"Cleaned {len(cleaned_asteroids)} asteroid records")
print(f"Extracted {len(close_approaches)} close approach events")

Cleaned 10000 asteroid records
Extracted 10000 close approach events


In [ ]:
# Save data

# Save cleaned asteroid list
with open("cleaned_asteroids.json", "w") as f:
    json.dump(cleaned_asteroids, f, default=str)

# Save close approach data
with open("close_approaches.json", "w") as f:
    json.dump(close_approaches, f, default=str)

print("Cleaned data saved to "cleaned_asteroids.json" and "close_approaches.json"")

Cleaned data saved to 'cleaned_asteroids.json' and 'close_approaches.json'


### Step 4: Insert NASA Asteroid Data into SQL

In [ ]:
import mysql.connector
import pandas as pd

In [ ]:

# Loading cleaned data from JSON
with open("cleaned_asteroids.json", "r") as f:
    cleaned_asteroids = json.load(f)

with open("close_approaches.json", "r") as f:
    close_approaches = json.load(f)

# MySQL connection settings
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="ShaYug",
    database="project_nasa_neo"
)

cursor = conn.cursor()

# Creating tables
cursor.execute("""
    CREATE TABLE IF NOT EXISTS asteroids (
        id BIGINT,
        name VARCHAR(255),
        absolute_magnitude_h FLOAT,
        estimated_diameter_min_km FLOAT,
        estimated_diameter_max_km FLOAT,
        is_potentially_hazardous_asteroid BOOLEAN
    )
""")

cursor.execute("""
    CREATE TABLE IF NOT EXISTS close_approach (
        neo_reference_id BIGINT,
        close_approach_date DATE,
        relative_velocity_kmph FLOAT,
        astronomical FLOAT,
        miss_distance_km FLOAT,
        miss_distance_lunar FLOAT,
        orbiting_body VARCHAR(100)
    )
""")

# Inserting into asteroids table
for a in cleaned_asteroids:
    cursor.execute("""
        INSERT INTO asteroids VALUES (%s, %s, %s, %s, %s, %s)
    """, (
        a["id"],
        a["name"],
        a["absolute_magnitude_h"],
        a["estimated_diameter_min_km"],
        a["estimated_diameter_max_km"],
        a["is_potentially_hazardous_asteroid"]
    ))

# Inserting into close_approach table
for ca in close_approaches:
    cursor.execute("""
        INSERT INTO close_approach VALUES (%s, %s, %s, %s, %s, %s, %s)
    """, (
        ca["neo_reference_id"],
        ca["close_approach_date"],
        ca["relative_velocity_kmph"],
        ca["astronomical"],
        ca["miss_distance_km"],
        ca["miss_distance_lunar"],
        ca["orbiting_body"]
    ))

#Commit and close
conn.commit()
conn.close()

print("Data successfully inserted into MySQL database!")


Data successfully inserted into MySQL database!
